In [ ]:
def main(datasources, start_date, end_date):
    """
    因子：日内流动性恶化率（Intraday Liquidity Deterioration）

    单因子评估结果：多空Sharpe=0.875，ModelScore=#6（1.454），入选率90%。

    文献依据：
        [1] Amihud (2002) - Illiquidity and Stock Returns: Cross-Section
            and Time-Series Effects, Journal of Financial Markets.
            经典非流动性度量：|收益率| / 成交额，每单位成交额引起的价格冲击。
        [2] Chordia, Roll & Subrahmanyam (2001) - Market Liquidity and
            Trading Activity, Journal of Finance.
            流动性变化（而非绝对水平）对收益有更强的预测力。

        本因子结合两篇文献：用1分钟数据计算Amihud指标（比日频更精准），
        并计算上午场vs下午场的流动性变化率（Chordia的变化量思路）。
        下午流动性相对上午恶化 → 知情交易者在消耗流动性 → 次日正向收益。
        与市值因子正交（BARRA风格剔除后仍有显著残差）。

    公式：illiq_pm / illiq_am - 1
          illiq = avg(|ret_1m| / amount)，分上下午分别计算
    方向：正向（流动性恶化越多 → 因子值越高 → 预期次日收益越高）

    参数:
        datasources (dict): {"bar1m": 分钟K线表名}
        start_date (str): 开始时间
        end_date (str):   结束时间

    返回:
        pd.DataFrame: ['date', 'instrument', 'factor']，不含 inf
    """
    import pandas as pd
    import numpy as np
    import dai

    bar1m = datasources["bar1m"]

    # ================================================================== #
    # 日内流动性恶化率因子                                                   #
    # 核心统计方法：Amihud非流动性度量的上下午差分                             #
    # 经济逻辑：下午流动性恶化代表知情交易者正在积累仓位，                        #
    #           信息不对称程度上升预示次日价格发现                              #
    # 技术实现：lag()窗口函数必须在独立CTE中完成，不能与聚合函数混用（DuckDB限制）#
    # ================================================================== #
    sql = f"""
    WITH cte_ret AS (
        -- 第一步：计算每分钟收益率（lag在独立层完成，避免与聚合混用）
        SELECT
            instrument,
            strftime(date, '%Y-%m-%d')  AS trading_day,
            strftime(date, '%H:%M')     AS hm,
            amount,
            close,
            lag(close, 1) OVER (
                PARTITION BY instrument, strftime(date, '%Y-%m-%d')
                ORDER BY date
            ) AS prev_close
        FROM {bar1m}
        WHERE close > 0 AND amount > 0
    ),
    cte_illiq AS (
        -- 第二步：分上下午聚合 Amihud 非流动性
        SELECT
            trading_day,
            instrument,
            avg(CASE
                WHEN hm < '12:00' AND prev_close > 0
                THEN abs(close / prev_close - 1) / (amount + 1e-8)
                ELSE NULL
            END) AS illiq_am,
            avg(CASE
                WHEN hm >= '13:00' AND prev_close > 0
                THEN abs(close / prev_close - 1) / (amount + 1e-8)
                ELSE NULL
            END) AS illiq_pm
        FROM cte_ret
        GROUP BY instrument, trading_day
    )
    -- 第三步：计算恶化率，过滤异常值
    SELECT
        CAST(trading_day AS DATETIME) AS date,
        instrument,
        (illiq_pm / NULLIF(illiq_am, 0) - 1) AS factor
    FROM cte_illiq
    WHERE illiq_am > 0
      AND illiq_pm > 0
      AND illiq_pm / NULLIF(illiq_am, 0) BETWEEN 0.1 AND 10
    """
    df = dai.query(sql, filters={"date": [start_date, end_date]}, compression=True).df()
    df["date"] = pd.to_datetime(df["date"])
    df["factor"] = pd.to_numeric(df["factor"], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["factor"])

    # 对齐中证1000成分股
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"])

    result = pd.merge(df, stk_pool, how="inner", on=["date", "instrument"])
    return result[["date", "instrument", "factor"]].reset_index(drop=True)


if __name__ == "__main__":
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    datasources = {
        "bar1m":     "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
    }
    start_date = "2024-01-01 00:00:00"
    end_date   = "2024-12-31 23:59:59"

    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )
